# 20.13 交給別人使用時，要一起帶走哪些東西？


交付助理時，要讓下一位學生拿到同一份底座、處理器、語音入口與生成設定。若用了LoRA才附修正；本版沒有開啟它。只拿到一份檔案卻不知搭哪個底座，無法知道重現的是哪位助理。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from hashlib import sha256

expected = b"base A, adapter B"
received = b"base A, adapter C"
print("期待內容指紋", sha256(expected).hexdigest()[:12])
print("收到內容指紋", sha256(received).hexdigest()[:12])
print("內容符合清單", sha256(expected).digest() == sha256(received).digest())

兩串人工bytes只有adapter名稱不同，完整指紋比較就False。畫面截短12字元只是方便讀；正式下載比較完整SHA-256與檔案大小。指紋確認內容配對，不保證回答正確。

選定配置是Qwen3-VL-2B-Instruct＋Whisper-large-v3-turbo，無LoRA。底座有2,127,532,032參數，ASR另有808,878,080，權重來源與工作各自列明。[執行核對](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/selected-final-test-audit-summary.json)和[ASR選版](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/asr-selection.json)保留固定配置。

能力卡每題同時核對內容、請求與正常結束。最後共178份回答，171份正常結束，7份在文字聊天組用滿384個新token而截斷；這7份仍留在13題分母裡。

| 用途 | 最後考卷結果 | 考了什麼，結果支持到哪裡 |
| --- | --- | --- |
| 自然照片的主要場景描述 | 25/42題通過 | 42張照片，各要求一至兩句繁中描述主要內容；不是把所有細節逐一列完。 |
| 自然照片的可見事實問答 | 58/84題通過 | 同42張照片各問兩題，包含物件、屬性、數量、動作與關係。當中動作題3/11、關係題21/29，已包含在84題裡。 |
| 有沒有清楚可見的中文字 | 18/18題通過 | 18張真實照片，10張有、8張沒有；只判有無，不代表讀出文字。 |
| 真實照片裡指定區域的逐字抄寫 | 8/10題通過 | 10張照片，先做NFKC字元形式正規化並移除全部空白，保留繁簡差異及正規化後的標點；不是整張照片所有文字的轉寫。 |
| 真實文字的指定讀序與分行 | 1/3題通過 | 三張照片，依問題選定文字、排列順序並保留要求的換行；文字與順序一起正確才算通過。 |
| 中文文字請求與聊天 | 2/13題通過 | 13題裡只有1題必須依前文作答，該題未通過；其中7題截斷。沒有驗收長多輪聊天。 |
| 真人語音轉寫 | 原樣CER 112/674＝16.62%；規約CER 96/661＝14.52% | 22段普通話真人朗讀：18段FLEURS句子、4段AISHELL問句；這是聽寫字元誤差率，不是聊天成功率。 |
| 同一語音問句，先給正確文字再回答 | 2/4題通過 | 把四段AISHELL問句的來源逐字稿送進聊天核心，檢查它收到正確請求時的回答。 |
| 同一語音問句，先辨識錄音再回答 | 2/4題通過 | 使用同四段錄音的實際ASR逐字稿，接進相同聊天核心，檢查整條語音問答路線。 |

42個描述與84個問答共用42張圖；字題也共用部分照片，兩條語音問答共用四個問題。因此178份回答不是178個獨立情境，不合成一個「什麼都會」的總分。結果和題型子集見[逐題得分](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/blind-review/scored/scores.json)與[子集](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/evaluate-37221188153/blind-review/descriptive-subsets.json)。

ASR原樣CER保留空白與標點；規約先做NFKC再去空白，仍留標點、大小寫和繁簡差異。四段AISHELL兩種CER皆0/42，兩路聊天仍各2/4，說明聽對之後仍要答對。動作3/11、讀序1/3與聊天2/13也應留在能力說明，不因某些讀字成功而蓋掉。

本版支持部分照片和中文文字任務，尚不足以當可靠通用助手。未測任意整頁、多欄、即興語音長對話，保持未驗收。已有官方權重與本課LoRA候選的作用都說清楚，才不會把沿用底座的能力記成從隨機權重學得。

下載配置與操作見[公開清單](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/public-release.json)、[20.2](https://birdhackor.github.io/tiny-perceptron-vlm/20.2.html)與[學生指引](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-student.html)。需要新候選時使用[資料說明](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-data.html)和[訓練指引](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-training.html)，另留驗證和最後題。推論配套與精確續訓狀態仍是不同包裹。
